# 01 - Data Ingestion
**Goal:** load the phishing data, clean it, split into train/test, save to disk.

Source: MongoDB if `MONGODB_URI` works, else the local CSV.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))   # so `import app` works

CSV_PATH = ROOT / "data" / "phisingData.csv"
OUT_DIR = ROOT / "Artifacts" / "notebooks"
OUT_DIR.mkdir(parents=True, exist_ok=True)
TARGET = "Result"

## 1. Load data
Try MongoDB first. Fall back to CSV if it is not reachable.

In [2]:
def load_from_mongo() -> pd.DataFrame:
    import pymongo
    from app.core.config import get_settings
    from app.constants import training_pipeline as tp
    client = pymongo.MongoClient(get_settings().mongodb_uri, serverSelectionTimeoutMS=3000)
    try:
        docs = list(client[tp.DATA_INGESTION_DATABASE_NAME][tp.DATA_INGESTION_COLLECTION_NAME].find())
    finally:
        client.close()
    df = pd.DataFrame(docs)
    return df.drop(columns=["_id"], errors="ignore")

try:
    df = load_from_mongo()
    if df.empty:
        raise ValueError("collection empty")
    source = "MongoDB"
except Exception as e:
    print(f"Mongo unavailable ({type(e).__name__}) -> using CSV")
    df = pd.read_csv(CSV_PATH)
    source = "CSV"

print("source:", source, "| shape:", df.shape)
df.head()

Mongo unavailable (ValidationError) -> using CSV
source: CSV | shape: (11055, 31)


,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,SSLfinal_State,Domain_registeration_length,Favicon,...,popUpWidnow,Iframe,age_of_domain,DNSRecord,web_traffic,Page_Rank,Google_Index,Links_pointing_to_page,Statistical_report,Result
0,-1,1,1,1,-1,-1,-1,-1,-1,1,...,1,1,-1,-1,-1,-1,1,1,-1,-1
1,1,1,1,1,1,-1,0,1,-1,1,...,1,1,-1,-1,0,-1,1,1,1,-1
2,1,0,1,1,1,-1,-1,-1,-1,1,...,1,1,1,-1,1,-1,1,0,-1,-1
3,1,0,1,1,1,-1,-1,-1,1,1,...,1,1,-1,-1,1,-1,1,-1,1,-1
4,1,0,-1,1,1,-1,1,1,-1,1,...,-1,1,-1,-1,0,-1,1,1,1,1


## 2. Quick checks

In [3]:
df = df.replace({"na": np.nan})          # NOTE: replace returns a new frame - assign it!
print("missing values:", int(df.isna().sum().sum()))
print("duplicate rows:", int(df.duplicated().sum()))
print("target counts:"); print(df[TARGET].value_counts())

missing values: 0
duplicate rows: 5206
target counts:
Result
 1    6157
-1    4898
Name: count, dtype: int64


## 3. Train / test split
`stratify` keeps the class ratio the same in both sets.

In [4]:
train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df[TARGET])
print(train.shape, test.shape)
print(train[TARGET].value_counts(normalize=True).round(3).to_dict())
print(test[TARGET].value_counts(normalize=True).round(3).to_dict())

(8844, 31) (2211, 31)
{1: 0.557, -1: 0.443}
{1: 0.557, -1: 0.443}


## 4. Save
Later notebooks read these files.

In [5]:
df.to_csv(OUT_DIR / "feature_store.csv", index=False)
train.to_csv(OUT_DIR / "train.csv", index=False)
test.to_csv(OUT_DIR / "test.csv", index=False)
print("saved to", OUT_DIR)

saved to /home/surya/ml-proj/Artifacts/notebooks
